In [1]:
import sys
# here, either reference to satclip repo goes, or can try and create git submodule 🤷‍♀️ 
# https://github.com/microsoft/satclip/tree/main
sys.path.append('/home/cbutsko/Desktop/cbutsko_experiments/satclip/satclip')

# general
import numpy as np
import pandas as pd
import itertools
import gc
import re
import torch
import glob
import rioxarray
import xarray as xr
import rasterio as rio
import os

# modeling
from catboost import CatBoostClassifier, Pool
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, precision_score, recall_score
from sklearn.model_selection import train_test_split
# from sklearn.utils.validation import check_is_fitted, check_array
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

# plotting and output
from tqdm.auto import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from cbutsko_utils import plot_confusion_matrix
import warnings
warnings.filterwarnings(action='ignore')
tqdm.pandas()

# SatClip libs
from load import get_satclip
from cbutsko_utils import prepare_satclip_embeddings

# processing utils
from cbutsko_utils import patch2feats, process_raw_features_input_df

# hierarchical clsfc libs
from hiclass import LocalClassifierPerParentNode, LocalClassifierPerNode
from cbutsko_utils import CatBoostClassifierWrapper, LocalClassifierPerNodeWrapper, LocalClassifierPerParentNodeWrapper

In [2]:
# read files with mappings to crop names and hierarchy

label_columns = ['cropland', 'landcover', 'cropgroup', 'croptype']

wc2ec_map = pd.read_csv('resources/wc2eurocrops_map.csv')
ec_map = pd.read_csv('resources/eurocrops_map_wcr_edition.csv', index_col='ec_code')

In [3]:
# read parquet files, assign hierarchical labels and do small clean-ups 
tpath = '/vitodata/worldcereal/features/benchmarking_features/calval/'
freq = 'monthly'

data_df = process_raw_features_input_df(
    '{}rawts-{}_calval.parquet'.format(tpath, freq),
    wc2ec_map,
    ec_map,
    label_columns,
    add_countries=True
    )

In [4]:
data_df.shape

(839023, 214)

In [5]:
# add pre-computed presto features 
# presto_df = pd.read_parquet('{}pretrainedprestofts-{}_calval.parquet'.format(tpath,freq))
presto_df = pd.read_parquet('{}prestofts-{}_calval.parquet'.format(tpath,freq))

presto_df.set_index('sample_id', inplace=True)
presto_emb_feats = [xx for xx in presto_df.columns if 'presto_ft' in xx]

data_df = pd.concat([data_df,presto_df[presto_emb_feats]], join='inner', axis=1)
del presto_df
gc.collect()

0

### Couple of distributions of training data

In [ ]:
tdf = data_df[['cropgroup_name','aez_zoneid']].value_counts().sort_index().reset_index()
tdf = tdf.pivot(index='aez_zoneid', columns='cropgroup_name', values='count')
tdf.fillna(0, inplace=True)
tdf.drop(columns=['not_crop','not_cereal'], inplace=True)
fig, ax = plt.subplots(figsize=(6,18))
sns.heatmap(ax=ax, data=tdf, cmap='Blues', linewidths=0.30, annot=False)

In [ ]:
tdf = data_df[['aez_zoneid','year']].value_counts().sort_index().reset_index()
tdf = tdf.pivot(index='aez_zoneid', columns='year', values='count')
tdf.fillna(0, inplace=True)
fig, ax = plt.subplots(figsize=(5,25))
sns.heatmap(ax=ax, data=tdf, cmap='Blues', linewidths=0.30, annot=False)

In [ ]:
tdf = data_df[~data_df['cropgroup_name'].isin(['not_crop','not_cereal'])][['cropgroup_name','year']]
tdf = tdf.value_counts().sort_index().reset_index()
tdf = tdf.pivot(index='cropgroup_name', columns='year', values='count')
tdf.fillna(0, inplace=True)
fig, ax = plt.subplots(figsize=(6,6))
sns.heatmap(ax=ax, data=tdf, cmap='Blues', linewidths=0.30, annot=True, fmt='n') 

### SatCLIP embeddings

1. Get the embeddings themselves for train samples

In [ ]:
# again, here goes either path to a model, or git submodule can be created 🤷‍♀️
model_path = '/home/cbutsko/Desktop/cbutsko_experiments/satclip-resnet18-l10.ckpt'
model = get_satclip(model_path, device='cpu')

In [ ]:
satclip_df = prepare_satclip_embeddings(model, data_df)
satclip_emb_feats = [xx for xx in satclip_df.columns if 'satclip_ft' in xx]
data_df = pd.concat([data_df,satclip_df[satclip_emb_feats]], join='inner', axis=1)

del satclip_df
gc.collect()

2. Fit PCA on (part of) data, transform SatCLIP embeddings into PCs and append them to main dataframes

In [ ]:
n_rows = data_df.shape[0]
n_pcs = 32
satclip_pca_feats = ['satclip_PC{}'.format(jj) for jj in range(n_pcs)]

satclip_pca_model = PCA(n_components=n_pcs)
scaler = StandardScaler()
scaler.fit(data_df[satclip_emb_feats].sample(n=int(0.7*n_rows)).values)
satclip_emb_norm = scaler.transform(data_df[satclip_emb_feats].values)
satclip_pca_model.fit(satclip_emb_norm[np.random.randint(n_rows, size=int(0.7*n_rows)),:])
gc.collect()
satclip_pca = satclip_pca_model.transform(satclip_emb_norm)
satclip_pca_df = pd.DataFrame(
    satclip_pca, 
    columns=satclip_pca_feats, 
    index=data_df.index)

data_df = pd.concat([data_df,satclip_pca_df], join='inner', axis=1)

del satclip_pca_df
gc.collect()

### Classification

In [6]:
# group feature names for easier use
n_ts = 12

optical12_feats = [xx for xx in data_df.columns if re.search(r'OPTICAL.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
sar12_feats = [xx for xx in data_df.columns if re.search(r'SAR.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
temp12_feats  = [xx for xx in data_df.columns if re.search(r'METEO-temp.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
prcp12_feats  = [xx for xx in data_df.columns if re.search(r'METEO-precip.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
dem_feats = ['DEM-alt-20m', 'DEM-slo-20m']
latlon_feats = ['lat','lon']
biome_feats = [xx for xx in data_df.columns if 'biome' in xx]

satclip_emb_feats = [xx for xx in data_df.columns if 'satclip_ft' in xx]
presto_emb_feats = [xx for xx in data_df.columns if 'presto_ft' in xx]

satclip_pca_feats = [xx for xx in data_df.columns if 'satclip_PC' in xx]

In [ ]:
for ts in range(n_ts):
    B04 = data_df['OPTICAL-B04-ts{}-10m'.format(ts)]
    B08 = data_df['OPTICAL-B08-ts{}-10m'.format(ts)]
    _ndvi = (B08 - B04) / (B08 + B04)
    data_df['NDVI-ts{}-10m'.format(ts)] = _ndvi
    del B04,B08,_ndvi
    gc.collect()

ndvi_feats = [xx for xx in data_df.columns if re.search(r'NDVI.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]

In [7]:
# minor ammendments for smooth processing
# delete 5 points that for some reason don't have country
data_df = data_df[data_df['country_code'].notna()]
# delete 1 point that belongs to the country with only 1 point
data_df = data_df[data_df['country_code']!='DMA']

In [8]:
data_df['is_maize'] = data_df['CROPTYPE_NAME']=='Maize'

In [20]:
# Define features to use, label, year and/or AEZ to leave out
# label_col = 'cropland_wc'
label_col = 'is_maize'
# features_list = presto_emb_feats
features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats

In [ ]:
data_df[
    (data_df['country_code']=='ESP')
    ][['ref_id','CROPTYPE_NAME']].value_counts().sort_index().iloc[:20]

In [ ]:
_ndvi_subset.shape

In [ ]:
country = 'ZMB'
crop = 'Annual cropland'
# ref_id = '2020_BR_LEM-MAR'
 
_ndvi_subset = data_df[
    (data_df['country_code']==country) &
    (data_df['CROPTYPE_NAME']==crop)
    # (~data_df['cropland_wc'])
    ]

ndvis = _ndvi_subset.sample(n=20)[ndvi_feats].transpose()
ndvis[ndvis.abs()>1] = 0

ndvis.plot(figsize=(10,5), legend=None)
plt.title('{}, {}'.format(crop, country))

In [ ]:
test_country = 'ESP'

X_trnval_df = data_df[data_df['country_code']!=test_country][features_list]
y_trnval_df = data_df[data_df['country_code']!=test_country][label_col]
X_tst_df = data_df[data_df['country_code']==test_country][features_list]
y_tst_df = data_df[data_df['country_code']==test_country][label_col]

# initialize and train the model
model = CatBoostClassifier(
    iterations=500, 
    depth=8,
    eval_metric='F1',
    learning_rate=0.3,
    l2_leaf_reg=100,
    verbose=50,
    random_seed=42,
    )

model.fit(X_trnval_df, y_trnval_df)
probs = model.predict_proba(X_tst_df)
pred = model.predict(X_tst_df).flatten()
if y_tst_df.nunique()==2:
    pred = np.array([xx=='True' for xx in pred])

# create dataframe with predictions and useful attributes
preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
preds_df.columns = ['true','pred']
preds_df['crop_prob'] = probs[:,1]
preds_df.set_index(data_df[data_df['country_code']==test_country].index, inplace=True)
attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code']
preds_df[attr_lst] = data_df[data_df['country_code']==test_country][attr_lst]
preds_df.to_csv('/home/cbutsko/Desktop/cbutsko_experiments/{}_predictions.csv'.format(test_country))

In [ ]:
preds_df.groupby(['ref_id','CROPTYPE_NAME'])['pred'].value_counts().sort_index().iloc[40:60]

In [ ]:
preds_df.groupby(['ref_id']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_cropland_pixels': xx['true'].sum(),
    'f1': f1_score(xx['true'], xx['pred']),
    'accuracy': (xx['true']==xx['pred']).sum()/xx['true'].count()
    }))

In [ ]:
res_fpath = '/home/cbutsko/Desktop/cbutsko_experiments/country_loo_res_presto_finetuned_feats_satclip_emb.csv'
min_points_per_country = 1000

countries_lst = data_df['country_code'].value_counts()
countries_lst = countries_lst[countries_lst>min_points_per_country].index

if os.path.isfile(res_fpath):
    country_loo_res = pd.read_csv(res_fpath)
    if 'Unnamed: 0' in country_loo_res.columns:
        country_loo_res.drop('Unnamed: 0', axis=1, inplace=True)
    computed_countries = country_loo_res['country_code'].nunique()
    countries_lst = countries_lst[computed_countries:]
else: 
    country_loo_res = pd.DataFrame()

pbar = tqdm(total=len(countries_lst))
for test_country in countries_lst:
    pbar.update(1)

    X_trnval_df = data_df[data_df['country_code']!=test_country][features_list]
    y_trnval_df = data_df[data_df['country_code']!=test_country][label_col]
    X_tst_df = data_df[data_df['country_code']==test_country][features_list]
    y_tst_df = data_df[data_df['country_code']==test_country][label_col]

    # initialize and train the model
    model = CatBoostClassifier(
        iterations=500, 
        depth=8,
        eval_metric='F1',
        learning_rate=0.3,
        l2_leaf_reg=100,
        verbose=0,
        random_seed=42,
        )

    model.fit(X_trnval_df, y_trnval_df)
    probs = model.predict_proba(X_tst_df)
    pred = model.predict(X_tst_df).flatten()
    if y_tst_df.nunique()==2:
        pred = np.array([xx=='True' for xx in pred])

    # create dataframe with predictions and useful attributes
    preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
    preds_df.columns = ['true','pred']
    preds_df['crop_prob'] = probs[:,1]
    preds_df.set_index(data_df[data_df['country_code']==test_country].index, inplace=True)
    attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code']
    preds_df[attr_lst] = data_df[data_df['country_code']==test_country][attr_lst]
    tres = preds_df.groupby(['ref_id','country_code']).apply(lambda xx: pd.Series({
        'n_pixels': xx['ref_id'].count(),
        'f1': f1_score(xx['true'], xx['pred'])
        })).reset_index()
    tres = tres[tres['country_code']==test_country].sort_values(by='n_pixels', ascending=False)

    country_loo_res = pd.concat((country_loo_res,tres), axis=0)
    country_loo_res.to_csv(res_fpath)

    gc.collect()

**Compute metrics on country-stratified random split**

In [16]:
trn_df, val_df = train_test_split(
    data_df,
    stratify=data_df['country_code'],
    test_size=0.3,
    random_state=42)

X_trnval_df = trn_df[features_list]
y_trnval_df = trn_df[label_col]
X_tst_df = val_df[features_list]
y_tst_df = val_df[label_col]

# initialize and train the model
model = CatBoostClassifier(
    iterations=500, 
    depth=8,
    eval_metric='F1',
    learning_rate=0.3,
    l2_leaf_reg=100,
    verbose=50,
    random_seed=42,
    )

model.fit(X_trnval_df, y_trnval_df)
probs = model.predict_proba(X_tst_df)
pred = model.predict(X_tst_df).flatten()
if y_tst_df.nunique()==2:
    pred = np.array([xx=='True' for xx in pred])

# create dataframe with predictions and useful attributes
preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
preds_df.columns = ['true','pred']
preds_df['crop_prob'] = probs[:,1]
preds_df.set_index(val_df.index, inplace=True)
attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code']
preds_df[attr_lst] = val_df[attr_lst]
tres = preds_df.groupby(['ref_id','country_code']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred'])
    })).reset_index()
# tres.to_csv('/home/cbutsko/Desktop/cbutsko_experiments/country_stratified_split_all_feats_no_emb.csv')

gc.collect()

0:	learn: 0.4055123	total: 224ms	remaining: 1m 51s
50:	learn: 0.6871472	total: 8.24s	remaining: 1m 12s
100:	learn: 0.7318619	total: 16.7s	remaining: 1m 5s
150:	learn: 0.7588803	total: 24.9s	remaining: 57.7s
200:	learn: 0.7798359	total: 33.3s	remaining: 49.5s
250:	learn: 0.7985247	total: 41.8s	remaining: 41.5s
300:	learn: 0.8116066	total: 50.1s	remaining: 33.1s
350:	learn: 0.8247211	total: 58.5s	remaining: 24.8s
400:	learn: 0.8347666	total: 1m 6s	remaining: 16.5s
450:	learn: 0.8467287	total: 1m 14s	remaining: 8.13s
499:	learn: 0.8561053	total: 1m 22s	remaining: 0us


0

In [17]:
# presto_emb_feats
print(classification_report(
    preds_df['true'], 
    preds_df['pred'], 
    target_names=['not_maize','maize'])) 

              precision    recall  f1-score   support

   not_maize       0.98      0.99      0.99    238422
       maize       0.83      0.66      0.73     13284

    accuracy                           0.97    251706
   macro avg       0.91      0.83      0.86    251706
weighted avg       0.97      0.97      0.97    251706



In [12]:
# optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats
print(classification_report(
    preds_df['true'], 
    preds_df['pred'], 
    target_names=['not_maize','maize'])) 

              precision    recall  f1-score   support

   not_maize       0.98      0.99      0.99    238422
       maize       0.85      0.71      0.78     13284

    accuracy                           0.98    251706
   macro avg       0.92      0.85      0.88    251706
weighted avg       0.98      0.98      0.98    251706



In [18]:
tres = preds_df.groupby(['country_code']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()

In [19]:
# presto_emb_feats
tres.sort_values(by='n_maize_pixels', ascending=False).iloc[:20]

,country_code,n_pixels,n_maize_pixels,f1,precision,recall
10,BEL,31067.0,3928.0,0.761182,0.831423,0.701884
163,USA,41290.0,2198.0,0.745889,0.824339,0.681074
7,AUT,31419.0,1633.0,0.685694,0.794355,0.603184
54,FRA,16741.0,1568.0,0.762478,0.856802,0.686862
4,ARG,5412.0,943.0,0.822928,0.885226,0.768823
161,UKR,3176.0,513.0,0.708197,0.805970,0.631579
49,ESP,19949.0,507.0,0.736127,0.864362,0.641026
26,CAN,6293.0,399.0,0.686610,0.795380,0.604010
116,NLD,1856.0,286.0,0.785321,0.826255,0.748252
40,DEU,2216.0,154.0,0.745247,0.899083,0.636364


In [14]:
# optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats
tres.sort_values(by='n_maize_pixels', ascending=False).iloc[:20]

,country_code,n_pixels,n_maize_pixels,f1,precision,recall
10,BEL,31067.0,3928.0,0.786371,0.838524,0.740326
163,USA,41290.0,2198.0,0.779284,0.832472,0.732484
7,AUT,31419.0,1633.0,0.732215,0.809948,0.668096
54,FRA,16741.0,1568.0,0.806564,0.891204,0.736607
4,ARG,5412.0,943.0,0.863014,0.934487,0.801697
161,UKR,3176.0,513.0,0.838912,0.905192,0.781676
49,ESP,19949.0,507.0,0.762955,0.865000,0.682446
26,CAN,6293.0,399.0,0.850543,0.928783,0.784461
116,NLD,1856.0,286.0,0.830389,0.839286,0.821678
40,DEU,2216.0,154.0,0.821918,0.869565,0.779221


In [21]:
X_tst_df.fillna(0, inplace=True)

In [23]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    verbose=1,
    n_jobs=-1,
    random_state=42,
    )

model.fit(X_trnval_df, y_trnval_df)
pred = model.predict(X_tst_df)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 192 tasks      | elapsed:  6.5min
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed: 15.4min
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed: 17.3min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    1.7s
[Parallel(n_jobs=4)]: Done 442 tasks      | elapsed:    3.8s
[Parallel(n_jobs=4)]: Done 500 out of 500 | elapsed:    4.3s finished


In [24]:
print(
"""
F1 score: {:.2f}
Precision: {:.2f}
Recall: {:.2f}
""".format(
    f1_score(y_tst_df.values, pred),
    precision_score(y_tst_df.values, pred),
    recall_score(y_tst_df.values, pred))
    )


F1 score: 0.50
Precision: 0.87
Recall: 0.35



### Spatial test

In [ ]:
# path to test patches
test_patches_path = '/home/cbutsko/Desktop/cbutsko_experiments/test_patches/'
presto_fts_path = '/vitodata/worldcereal/data/openeo/inputs_presto/preprocessed_out/presto_embeddings/finetuned/'
test_patches_files = glob.glob('{}*.nc'.format(test_patches_path))
presto_fts_test_patches_files = glob.glob('{}*.nc'.format(presto_fts_path))

In [ ]:
patch_ind = 2

test_patch_fpath = test_patches_files[patch_ind]
test_patch = rioxarray.open_rasterio(
    test_patch_fpath, 
    decode_times=False)
# test_patch = test_patch.rio.reproject('EPSG:4326')

# presto_fts_test_patch_fpath = [xx for xx in presto_fts_test_patches_files if test_patch_fpath.split('/')[-1].split('_')[0] in xx][0]
# presto_fts_test_patch = rioxarray.open_rasterio(
#     presto_fts_test_patch_fpath,
#     decode_times=False)

ts_ind = 6
test_patch_rgb = np.dstack((
    test_patch.B04.values[ts_ind,:,:],
    test_patch.B03.values[ts_ind,:,:],
    test_patch.B02.values[ts_ind,:,:]))

test_patch_ndvi = (test_patch.B08.values[ts_ind,:,:] - test_patch.B04.values[ts_ind,:,:]) / (test_patch.B08.values[ts_ind,:,:] + test_patch.B04.values[ts_ind,:,:])
test_patch_fpath

In [ ]:
test_patch_df_feats = patch2feats(test_patch, features_list)

# test_patch_df_feats = presto_fts_test_patch.values.reshape(128,-1).swapaxes(0,1)
# test_patch_df_feats = pd.DataFrame(test_patch_df_feats, columns=presto_emb_feats)

test_patch_probs = model.predict_proba(test_patch_df_feats)[:,1]
test_patch_pred = model.predict(test_patch_df_feats).flatten()
test_patch_pred = np.array([xx=='True' for xx in test_patch_pred])

In [ ]:
fig = plt.figure(figsize=(12,8))

fig.add_subplot(2, 3, 1)
plt.imshow(test_patch.worldcereal_cropland.values[0,:,:])
plt.title('WorldCereal Mask')

fig.add_subplot(2, 3, 2)
plt.imshow(test_patch_rgb, norm='linear')
plt.title('RGB')

fig.add_subplot(2, 3, 3)
plt.imshow(test_patch_ndvi)
plt.title('NDVI')

fig.add_subplot(2, 3, 4)
plt.imshow(test_patch_pred.reshape(test_patch['x'].shape[0], test_patch['y'].shape[0]))
plt.title('Raw Feats Prediction')

fig.add_subplot(2, 3, 5)
plt.imshow(test_patch_probs.reshape(test_patch['x'].shape[0], test_patch['y'].shape[0]))
plt.title('Raw Feats Probs')

### Hierarchical classification

In [ ]:
# pretend that crop/not_crop is solved
# this is also necessary, as now kernel keeps crashing on full dataset 😢
data_df = data_df[data_df['cropland_wc']==1]
# label_columns = label_columns[1:]
label_columns.remove('cropland')
label_columns.remove('cropgroup')

In [ ]:
# create train/test datasets
stratification_col = 'croptype'
test_size = 0.3

trnval_df, tst_df = train_test_split(
    data_df,
    stratify=data_df[stratification_col],
    test_size=test_size,
    random_state=42)

X_trn_df = trnval_df[features_list]
y_trn_df = trnval_df[label_columns]

X_tst_df = tst_df[features_list]
y_tst_df = tst_df[label_columns]

In [ ]:
cb_clsfr = CatBoostClassifierWrapper(
    iterations=2000, 
    depth=8,
    eval_metric='F1',
    learning_rate=0.3,
    l2_leaf_reg=100,
    verbose=50,
    random_seed=42,
)

classifier = LocalClassifierPerNode(
    local_classifier=cb_clsfr,
    n_jobs=-1,
)

classifier.fit(X_trn_df, y_trn_df)
# pred, probs = classifier.predict_proba(X_tst_df)
pred = classifier.predict(X_tst_df)
pred = pred.astype('int32')

In [ ]:
# assess predictions at a level
level_ind = -1

level_true = y_tst_df[label_columns[level_ind]]
level_pred = pred[:,level_ind]

# map to human names
tmap = ec_map[['{}_label'.format(label_columns[level_ind]),'{}_name'.format(label_columns[level_ind])]].drop_duplicates()
tmap = dict(tmap.to_numpy())
level_true = level_true.map(tmap)
level_pred = [*map(tmap.get, level_pred)]

In [ ]:
print(classification_report(
    level_true, 
    level_pred, 
    target_names=np.sort(level_true.unique()))) 

In [ ]:
cm = confusion_matrix(level_true, level_pred)
plot_confusion_matrix(cm, np.unique(level_true))

**Compare to flat classifier**

In [ ]:
label_col = label_columns[-1]

# compare to performance on random split
trn_df, val_df = train_test_split(
    trnval_df,
    stratify=trnval_df['croptype'],
    test_size=0.3,
    random_state=42)

X_trn_df = trn_df[features_list]
y_trn_df = trn_df[label_col]
X_val_df = val_df[features_list]
y_val_df = val_df[label_col]

In [ ]:
# initialize and train the model
model = CatBoostClassifier(
    iterations=2000, 
    depth=8,
    eval_metric='TotalF1',
    learning_rate=0.3,
    l2_leaf_reg=100,
    verbose=50,
    random_seed=42,
    )

model.fit(X_trn_df, y_trn_df, eval_set=(X_val_df, y_val_df), early_stopping_rounds=200)
probs = model.predict_proba(X_tst_df)
pred = model.predict(X_tst_df).flatten()

In [ ]:
true = y_tst_df.reset_index(drop=True)[label_col]
pred = pd.Series(model.predict(X_tst_df).flatten())

# # map to human names
# tmap = ec_map[['{}_label'.format(label_columns[-1]),'{}_name'.format(label_columns[-1])]].drop_duplicates()
# tmap = dict(tmap.to_numpy())
# true = true.map(tmap)
# pred = pred.map(tmap)
# pred.fillna('unknown', inplace=True)
# pred = pd.Series([*map(tmap.get, pred)])

In [ ]:
print(classification_report(
    true, 
    pred, 
    # target_names=np.sort(true.unique())
    )) 

In [ ]:
cm = confusion_matrix(true, pred)
plot_confusion_matrix(cm, np.unique(list(np.unique(true)) + list(np.unique(pred))))

**Using probabilities to train a simple meta learner**

In [ ]:
_, probs_trn = classifier.predict_proba(X_trn_df)
_, probs_tst = classifier.predict_proba(X_tst_df)

probs_features_trn = np.concatenate((
    probs_trn['l0'],
    probs_trn['l1']['1101']), axis=1)
probs_features_tst = np.concatenate((
    probs_tst['l0'], 
    probs_tst['l1']['1101']), axis=1)

In [ ]:
logreg_meta_learner_probs_model = LogisticRegression(random_state=42).fit(
    probs_features_trn, 
    y_trn_df[label_columns[-1]].values)

logreg_meta_learner_pred = logreg_meta_learner_probs_model.predict(probs_features_tst) 
tmap = ec_map[['{}_label'.format(label_columns[-1]),'{}_name'.format(label_columns[-1])]].drop_duplicates()
tmap = dict(tmap.to_numpy())
logreg_meta_learner_pred = [*map(tmap.get, logreg_meta_learner_pred)]

In [ ]:
print(classification_report(
    level_true, 
    logreg_meta_learner_pred, 
    target_names=np.sort(level_true.unique())))

In [ ]:
preds_df = pd.DataFrame([level_true.values, level_pred, logreg_meta_learner_pred]).transpose()
preds_df.columns = ['true','hiclass_pred','logreg_meta_pred']

In [ ]:
crops_lst = ['arable_crops', 'not_cereal', 'cereals', 'unspecified_cereal', 'wheat', 'barley', 'rye', 'oats', 'maize', 'millet_sorghum', 'rice']
ii = 442
['{}: {}'.format(ii,jj) for ii, jj in zip(crops_lst, np.round(probs_features_tst[ii,:], 2))]

In [ ]:
preds_df[
    (preds_df['hiclass_pred']==preds_df['true']) &
    (preds_df['logreg_meta_pred']!=preds_df['true'])
    ][['true','logreg_meta_pred']].value_counts().sort_index()